In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the dataset
path = "F:/Users/MSR/Downloads/decoder_DATA.csv"
df =pd.read_csv(path, sep=" ")

# Display DAtaset info
print(df.head())

print(df.info())


                sequence
36 37 38 39 40        41
19 20 21 22 23        24
39 40 41 42 43        44
14 15 16 17 18        19
25 26 27 28 29        30
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 50 entries, (36, 37, 38, 39, 40) to (37, 38, 39, 40, 41)
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sequence  50 non-null     int64
dtypes: int64(1)
memory usage: 7.0 KB
None


In [ ]:

Cell 2: Prepare Sequence of Numbers Dataset

Reconstructs the full 6-number sequences by combining the MultiIndex tuples with the sequence column, then creates progressive input/target pairs for next-number prediction.

In [5]:
# Combine the 5 numbers in the MultiIndex with the 6th number in the 'sequence' column
sequences = [list(idx) + [val] for idx, val in zip(df.index, df["sequence"])]

seq_len = len(sequences[0])       # 6
max_input_len = seq_len - 1       # 5

# Shift numbers by +1 so 0 is reserved exclusively for padding
all_numbers = [num for seq in sequences for num in seq]
vocab_size = max(all_numbers) + 2  # +1 for 0-indexing, +1 for padding token (0)

X, y = [], []
for seq in sequences:
    shifted = [n + 1 for n in seq]
    for i in range(1, len(shifted)):
        X.append(shifted[:i])
        y.append(shifted[i])

# Pre-pad inputs so the latest number is always aligned at the last timestep
X = keras.preprocessing.sequence.pad_sequences(X, maxlen=max_input_len, padding="pre")
y = np.array(y)

print(f"Sample reconstructed sequence: {sequences[0]}")
print(f"X shape: {X.shape}, y shape: {y.shape}")


Sample reconstructed sequence: [36, 37, 38, 39, 40, 41]
X shape: (250, 5), y shape: (250,)


In [ ]:

Cell 3: Build Transformer Block (Keras Functional API)

Builds the embedding layers and causal Transformer block using the Keras Functional API.

In [6]:
def build_transformer_model(vocab_size, max_len, embed_dim=32, num_heads=2, ff_dim=64, dropout_rate=0.1):
    inputs = keras.Input(shape=(max_len,), dtype="int32")

    # Token and Positional Embeddings
    positions = tf.range(start=0, limit=max_len, delta=1)
    token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)(positions)
    x = token_emb + pos_emb

    # Transformer Block (Functional API)
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim
    )(x, x, use_causal_mask=True)
    attn_output = layers.Dropout(dropout_rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(dropout_rate)(ffn_output)
    out2 = layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

    # Predict next number from the last position
    outputs = layers.Dense(vocab_size, activation="softmax")(out2[:, -1, :])

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_transformer_model(vocab_size=vocab_size, max_len=max_input_len)
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 5)]          0           []                               
                                                                                                  
 embedding (Embedding)          (None, 5, 32)        1536        ['input_1[0][0]']                
                                                                                                  
 tf.__operators__.add (TFOpLamb  (None, 5, 32)       0           ['embedding[0][0]']              
 da)                                                                                              
                                                                                                  
 multi_head_attention (MultiHea  (None, 5, 32)       8416        ['tf.__operators__.add[0][0]'

In [ ]:

Cell 4: Fit the Model to Predict the Next Number

Trains the model on the prepared dataset.

In [7]:
history = model.fit(
    X, y,
    batch_size=16,
    epochs=120,
    verbose=1
)


Epoch 1/120
16/16 [==============================] - 2s 8ms/step - loss: 2.9774 - accuracy: 0.3960
Epoch 2/120
16/16 [==============================] - 0s 7ms/step - loss: 0.6967 - accuracy: 0.9920
Epoch 3/120
16/16 [==============================] - 0s 7ms/step - loss: 0.2249 - accuracy: 1.0000
Epoch 4/120
16/16 [==============================] - 0s 8ms/step - loss: 0.0969 - accuracy: 1.0000
Epoch 5/120
16/16 [==============================] - 0s 9ms/step - loss: 0.0567 - accuracy: 1.0000
Epoch 6/120
16/16 [==============================] - 0s 8ms/step - loss: 0.0381 - accuracy: 1.0000
Epoch 7/120
16/16 [==============================] - 0s 9ms/step - loss: 0.0288 - accuracy: 1.0000
Epoch 8/120
16/16 [==============================] - 0s 8ms/step - loss: 0.0225 - accuracy: 1.0000
Epoch 9/120
16/16 [==============================] - 0s 8ms/step - loss: 0.0190 - accuracy: 1.0000
Epoch 10/120
16/16 [==============================] - 0s 7ms/step - loss: 0.0158 - accuracy: 1.0000
Epoch 11/

In [ ]:

Cell 5: Start with a Single Number and Generate the Rest One by One

Takes a single starting number and autoregressively predicts each subsequent number until a 6-number sequence is built.

In [11]:
def generate_sequence(model, start_number, total_length=6):
    generated = [start_number]

    for _ in range(total_length - 1):
        shifted_seq = [n + 1 for n in generated]
        padded_input = keras.preprocessing.sequence.pad_sequences(
            [shifted_seq], maxlen=max_input_len, padding="pre"
        )

        preds = model.predict(padded_input, verbose=0)
        next_token = int(np.argmax(preds)) - 1
        generated.append(next_token)

    return generated

# # Generate sequences starting from single numbers
# for seed in:
#     print(f"Start: {seed:2d} -> Generated: {generate_sequence(model, seed)}")

# # Generate sequences starting from single numbers
# for seed in:
#     print(f"Start: {seed:2d} -> Generated: {generate_sequence(model, seed)}")

# Generate sequences starting from single numbers
seeds = (1, 14, 25, 36)
for seed in seeds:
    print(f"Start: {seed:2d} -> Generated: {generate_sequence(model, seed)}")




Start:  1 -> Generated: [1, 2, 3, 4, 5, 6]
Start: 14 -> Generated: [14, 15, 16, 17, 18, 19]
Start: 25 -> Generated: [25, 26, 27, 28, 29, 30]
Start: 36 -> Generated: [36, 37, 38, 39, 40, 41]
